# 04 — CatBoost + SHAP Feature Attribution

Gradient-boosted tree model with 5-fold cross-validation hyperparameter tuning.
SHAP values decompose each prediction into additive feature contributions.

| | |
|---|---|
| **Inputs** | `data/processed/merged_analysis.csv` |
| **Outputs** | `output/figures/shap_beeswarm.png`, `output/figures/shap_bar.png` |

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from catboost import CatBoostRegressor, Pool, cv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PROC = Path("../data/processed")
FIG_OUT   = Path("../output/figures")
FIG_OUT.mkdir(parents=True, exist_ok=True)

GREEN = "#00693E"
DARK  = "#1a1a2e"

# ── Feature list ──────────────────────────────────────────────────────────────
FEATURES = ["ideology", "climate_concern", "education", "income",
            "age", "coal_share"]
TARGET   = "re_support"

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PROC / "merged_analysis.csv")
FEATURES = [c for c in FEATURES if c in df.columns]

X = df[FEATURES]
y = df[TARGET]
print(f"Loaded: X={X.shape}  |  y mean={y.mean():.3f}")

In [ ]:
# ── Train/test split (80/20) ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

In [ ]:
# ── 5-fold cross-validation to tune iterations ────────────────────────────────
train_pool = Pool(X_train, y_train)

cv_params = {
    "iterations": 500,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "RMSE",
    "verbose": False,
    "random_seed": 42
}

cv_results = cv(
    pool=train_pool,
    params=cv_params,
    fold_count=5,
    early_stopping_rounds=50,
    verbose=False
)

best_iter = cv_results["test-RMSE-mean"].idxmin()
best_rmse = cv_results["test-RMSE-mean"].min()
print(f"Best iteration: {best_iter}  |  CV RMSE: {best_rmse:.4f}")

In [ ]:
# ── Train final model on best iteration ──────────────────────────────────────
model = CatBoostRegressor(
    iterations=best_iter,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2   = r2_score(y_test, y_pred)
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test R²:   {test_r2:.4f}")

In [ ]:
# ── SHAP values ───────────────────────────────────────────────────────────────
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print("SHAP values computed.")
print(f"Shape: {shap_values.shape}")

In [ ]:
# ── Figure 1: SHAP beeswarm plot ──────────────────────────────────────────────
plt.figure(figsize=(9, 5))
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=FEATURES,
    plot_type="dot",
    show=False,
    max_display=len(FEATURES)
)
plt.title("SHAP Feature Attribution: Renewable Energy Support",
          fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(FIG_OUT / "shap_beeswarm.png", bbox_inches="tight")
plt.show()
print("Saved: shap_beeswarm.png")

In [ ]:
# ── Figure 2: SHAP mean absolute bar chart ────────────────────────────────────
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
colors = [GREEN if f != "coal_share" else "#C8E6C9" for f in mean_shap.index]
ax.barh(mean_shap.index, mean_shap.values, color=colors, edgecolor="white")
ax.set_xlabel("Mean |SHAP value|", fontsize=11)
ax.set_title("Feature Importance (Mean Absolute SHAP)", fontsize=13, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_OUT / "shap_bar.png")
plt.show()
print("Saved: shap_bar.png")

print("\nFeature ranking by SHAP importance:")
print(mean_shap.sort_values(ascending=False).round(4))